# 03 — LEAR robustness across regimes (2022 H2, 2023, 2024)

Third milestone of sub-objective 3.1. Once the LEAR port (notebook 02)
has shown it beats the seasonal naive on a single June-2024 fortnight,
the next question is whether the same model survives in *other*
market regimes — different years, a crisis period, a pooled multi-year
test set.

**Setup**
- Same model code as notebook 02 (`LEAR` from
  `src/mibel_forecasting/models/lear.py`). No model logic is changed
  in this notebook.
- Three temporal regimes, plus a pooled view:
  - **2022 (H2, post-invasion crisis)** — `2022-07-01 → 2022-12-31`,
    `train_size = 180D`. Pre-2022 ESIOS history is not available in
    this repo's local cache, so 2022 carries an asymmetrically shorter
    training window. Discussed explicitly in §3.
  - **2023 (full year, post-crisis recovery)** —
    `2023-01-01 → 2023-12-31`, `train_size = 365D` (canonical Lago).
  - **2024 (full year, high-solar regime)** —
    `2024-01-01 → 2024-12-31`, `train_size = 365D`.
  - **2023-2024 pooled** — concatenation of the two annual runs above.
- Four models in each regime: seasonal naive (lag = 7d), LEAR ar-only
  (103 features, no exogenous), LEAR demand+wind (the Lago canonical
  247-feature configuration), LEAR demand+solar+wind (319 features).
- Daily recalibration in every regime (`test_size = 1D`, `step = 1D`).

**Execution architecture.** The three regimes are independent
backtests, so they run in parallel via
`concurrent.futures.ProcessPoolExecutor` with at most three workers.
Inside each worker the four models run sequentially — we do **not**
parallelize inside `rolling_forecast` and we do **not** parallelize
by day, so model logic and reproducibility are unchanged. The
parallelism boundary lives at the regime level only.

A **smoke test** (§5) reproduces the canonical notebook-02 numbers
(`naive MAE ≈ 40.568` and `LEAR demand+wind rMAE ≈ 0.394` on
2024-06-01..14) through the parallel runner before committing to the
heavy three-regime run. A divergence stops execution.

**Validity preconditions.** Every number in this notebook is valid
only under the conventions described in §9. Re-read §9 before acting
on any rMAE figure.


In [1]:
from __future__ import annotations
import os
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

from mibel_forecasting.data.loaders import load_dam_panel
from mibel_forecasting.evaluation._robustness import MODEL_NAMES, run_regime

DIAG = Path('../reports/diagnostics').resolve()
DIAG.mkdir(parents=True, exist_ok=True)
CSV_PATH      = DIAG / 'lear_robustness_2026_05.csv'
MD_PATH       = DIAG / 'lear_robustness_2026_05.md'
COVERAGE_PATH = DIAG / 'lear_robustness_dropped_days_2026_05.csv'

PANEL_RANGE = {'panel_start': '2022-01-01', 'panel_end': '2024-12-31'}


## 1. Load panel preview

Load the panel once in the main process to confirm shape and tz. The
parallel workers reload it themselves — `load_dam_panel` hits the
local ESIOS cache (no API calls) and is fast.


In [2]:
df = load_dam_panel(
    start=PANEL_RANGE['panel_start'], end=PANEL_RANGE['panel_end']
).dropna()
print(f'shape={df.shape}, tz={df.index.tz}')
print(f'range: {df.index.min()} -> {df.index.max()}')
df.head(2)


shape=(25843, 14), tz=UTC
range: 2022-01-01 00:00:00+00:00 -> 2024-12-31 22:00:00+00:00


,price_es,price_pt,price_fr,es_demand_fc,fr_demand_fc,es_solar_fc,es_wind_fc,fr_solar_fc,fr_wind_fc,fr_nuclear_avail,ntc_es_fr,ntc_fr_es,ttf_eur_mwh,co2_eur_t
datetime,,,,,,,,,,,,,,
2022-01-01 00:00:00+00:00,114.90,114.90,78.48,21532.0,49162.5,0.0,6427.0,0.0,2594.63,39185.0,2543.0,3283.0,73.706668,80.881248
2022-01-01 01:00:00+00:00,113.87,113.87,85.16,20068.0,47550.0,0.0,6463.0,0.0,2756.01,39078.0,2543.0,3283.0,73.706668,80.881248


## 2. Validity preconditions

Every rMAE figure in §6 and §8 is valid only under:

- **UTC internal convention** (commit `b38b456`). Day boundaries
  follow the UTC calendar, not Europe/Madrid wall-clock. Pre-migration
  numbers are not directly comparable. See
  `reports/diagnostics/utc_migration_2026_05.md`.
- **Realised targets masked before `predict`** (commit `00a0aaf`).
  `rolling_forecast` NA-s the realised target column before handing
  the test slice to any model; `LEAR.predict` further sources its
  price-lag block exclusively from training history.
- **Partial test days skipped uniformly** (commits `89cd2db`,
  `3d3a912`). A calendar day with fewer than 24 hours in the panel is
  excluded from prediction across **every** LEAR configuration. See
  §5 for the explicit dropped-day diagnostic.
- **Day-ahead exogenous variables** — `es_demand_fc`, `es_wind_fc`
  and `es_solar_fc` are documented as forecasts published before the
  DAM gate-closure in
  `reports/diagnostics/exogenous_audit_2026_05.md`. Upstream
  publication timestamps have not been independently re-pulled.


## 3. Regimes — and the 2022 caveats

The 2022 calendar year has two non-trivial structural factors layered
on top of each other:

1. **Crisis-energy regime.** TTF gas spiked from ~80 €/MWh in February
   to ~340 €/MWh in late August 2022 after the invasion of Ukraine
   and the Nord Stream sabotage. This propagated directly into the
   Iberian DAM through CCGT marginal generation.
2. **Pre-rule-change MIBEL.** The 'Iberian exception' gas-price cap
   (RD-Ley 10/2022) was in force from June 2022 onwards and shaved
   the DAM price below the uncapped marginal cost of gas. This is a
   separate change from the 2023-03-04 CID time-of-arrival →
   price-priority rule; both apply to 2022 data but only the gas cap
   touches DAM directly.

These effects act in opposite directions on the realised DAM level
and are not separable on this notebook's data. The 2022 numbers below
should therefore be read as *robustness diagnostics*, not as a clean
head-to-head with 2023/2024. The 2023 and 2024 regimes are
post-crisis and post-rule-change and are comparable to each other on
identical terms.

**Methodological asymmetry: `train_size`.** 2022 uses a 180-day
training window because no ESIOS price history before 2022-01-01 is
available in this repo's local cache; 2023 and 2024 use the canonical
365-day Lago window. The shorter 2022 window will tend to *help* LEAR
(less stale history during the crisis) so any 2022 weakness versus
the naive is conservative, not optimistic.


## 4. Regime specifications

A `spec` is a plain dict (picklable) so workers receive the full
configuration through `ProcessPoolExecutor.map`.


In [3]:
MODELS_LIST = list(MODEL_NAMES)
ORDER = ['2022 (H2, crisis)', '2023 (full year)', '2024 (full year)', '2023-2024 pooled']

REGIME_SPECS = [
    {
        'regime':     '2022 (H2, crisis)',
        'start':      '2022-07-01',
        'end':        '2022-12-31',
        'train_size': '180D',
        'models':     MODELS_LIST,
        **PANEL_RANGE,
    },
    {
        'regime':     '2023 (full year)',
        'start':      '2023-01-01',
        'end':        '2023-12-31',
        'train_size': '365D',
        'models':     MODELS_LIST,
        **PANEL_RANGE,
    },
    {
        'regime':     '2024 (full year)',
        'start':      '2024-01-01',
        'end':        '2024-12-31',
        'train_size': '365D',
        'models':     MODELS_LIST,
        **PANEL_RANGE,
    },
]
REGIME_SPECS


[{'regime': '2022 (H2, crisis)',
  'start': '2022-07-01',
  'end': '2022-12-31',
  'train_size': '180D',
  'models': ['naive',
   'LEAR ar-only',
   'LEAR demand+wind',
   'LEAR demand+solar+wind'],
  'panel_start': '2022-01-01',
  'panel_end': '2024-12-31'},
 {'regime': '2023 (full year)',
  'start': '2023-01-01',
  'end': '2023-12-31',
  'train_size': '365D',
  'models': ['naive',
   'LEAR ar-only',
   'LEAR demand+wind',
   'LEAR demand+solar+wind'],
  'panel_start': '2022-01-01',
  'panel_end': '2024-12-31'},
 {'regime': '2024 (full year)',
  'start': '2024-01-01',
  'end': '2024-12-31',
  'train_size': '365D',
  'models': ['naive',
   'LEAR ar-only',
   'LEAR demand+wind',
   'LEAR demand+solar+wind'],
  'panel_start': '2022-01-01',
  'panel_end': '2024-12-31'}]

## 5. Smoke test — reproduce notebook 02 through the parallel runner

Before launching the heavy three-regime run, reproduce the canonical
notebook-02 numbers on the same fortnight, through the same
`run_regime` entry point that the parallel pool will call. Tolerances
are deliberately generous — different versions of the panel
(.dropna over a wider range) and a different LEAR partial-day rule
can shift the figures slightly, but not by more than a few percent.

If the smoke test fails the cell raises and the rest of the notebook
does not execute.


In [4]:
SMOKE_TOL_REL = 0.03  # 3 % relative

smoke_spec = {
    'regime':     'smoke (nb02 reference)',
    'start':      '2024-06-01',
    'end':        '2024-06-14',
    'train_size': '365D',
    'models':     MODELS_LIST,
    **PANEL_RANGE,
}

t0 = time.time()
smoke_result = run_regime(smoke_spec)
print(f'smoke run_regime done in {time.time()-t0:.1f}s')

# Extract metrics
smoke_metrics = {m['model']: m for m in smoke_result['metrics']}
smoke_naive_mae    = smoke_metrics['naive']['MAE (EUR/MWh)']
smoke_dw_rmae      = smoke_metrics['LEAR demand+wind']['rMAE vs naive']

REF_NAIVE_MAE = 40.568   # from notebook 02 §3
REF_DW_RMAE   = 0.394    # from notebook 02 §3

def _rel(observed, expected):
    return abs(observed - expected) / expected if expected else float('inf')

dn = _rel(smoke_naive_mae, REF_NAIVE_MAE)
dr = _rel(smoke_dw_rmae,   REF_DW_RMAE)

print(f'naive MAE         {smoke_naive_mae:8.4f}  ref {REF_NAIVE_MAE:8.4f}  rel diff {dn*100:5.2f}%')
print(f'LEAR d+w rMAE     {smoke_dw_rmae:8.4f}  ref {REF_DW_RMAE:8.4f}  rel diff {dr*100:5.2f}%')

if dn > SMOKE_TOL_REL or dr > SMOKE_TOL_REL:
    raise RuntimeError(
        f'Smoke test failed: naive MAE rel diff {dn*100:.2f}% / '
        f'LEAR d+w rMAE rel diff {dr*100:.2f}% — tolerance was {SMOKE_TOL_REL*100:.0f}%. '
        'Halting before the full parallel run; investigate before proceeding.'
    )
print('SMOKE OK — proceeding to the full parallel run.')


smoke run_regime done in 124.1s
naive MAE          40.5678  ref  40.5680  rel diff  0.00%
LEAR d+w rMAE       0.3936  ref   0.3940  rel diff  0.10%
SMOKE OK — proceeding to the full parallel run.


## 6. Parallel backtests

Three regime backtests dispatched to a `ProcessPoolExecutor` with at
most three workers. Each worker reloads the panel and runs the four
models for one regime sequentially.


In [5]:
t0 = time.time()
n_workers = min(3, os.cpu_count() or 1)
print(f'launching {n_workers} workers for {len(REGIME_SPECS)} regimes...')

with ProcessPoolExecutor(max_workers=n_workers) as ex:
    results = list(ex.map(run_regime, REGIME_SPECS))

dt = time.time() - t0
print(f'all three regimes done in {dt:.1f}s ({dt/60:.1f} min wall)')
for r in results:
    n_hours = {m['model']: m['n_hours'] for m in r['metrics']}
    print(f'  {r["regime"]:22s}  ' + '  '.join(f'{k}={v}' for k,v in n_hours.items()))


launching 3 workers for 3 regimes...


all three regimes done in 4228.0s (70.5 min wall)
  2022 (H2, crisis)       naive=4272  LEAR ar-only=4272  LEAR demand+wind=4272  LEAR demand+solar+wind=4272
  2023 (full year)        naive=8663  LEAR ar-only=8663  LEAR demand+wind=8663  LEAR demand+solar+wind=8663
  2024 (full year)        naive=8591  LEAR ar-only=8591  LEAR demand+wind=8591  LEAR demand+solar+wind=8591


## 7. Aggregate results — and derive the pooled regime


In [6]:
forecasts: dict[tuple[str,str], pd.DataFrame] = {}
metric_rows: list[dict] = []
coverage_rows: list[dict] = []
for r in results:
    for model_name, f in r['forecasts'].items():
        forecasts[(r['regime'], model_name)] = f
    metric_rows.extend(r['metrics'])
    coverage_rows.extend(r['coverage'])

# Pooled 2023-2024 is derived; no extra LEAR fits.
pooled_label = '2023-2024 pooled'
for model_name in MODELS_LIST:
    pooled = pd.concat([
        forecasts[('2023 (full year)', model_name)],
        forecasts[('2024 (full year)', model_name)],
    ])
    forecasts[(pooled_label, model_name)] = pooled

# Pooled metrics
from mibel_forecasting.evaluation.metrics import mae, smape
from mibel_forecasting.evaluation.dm_test import diebold_mariano
naive_pool = forecasts[(pooled_label, 'naive')]
naive_pool_mae = mae(naive_pool['y_true'], naive_pool['y_pred'])
for model_name in MODELS_LIST:
    f = forecasts[(pooled_label, model_name)]
    m = mae(f['y_true'], f['y_pred'])
    s = smape(f['y_true'], f['y_pred'])
    r = m / naive_pool_mae if naive_pool_mae else float('nan')
    if model_name == 'naive':
        dm_stat, dm_pval, nw_lag = float('nan'), float('nan'), float('nan')
    else:
        dm = diebold_mariano(f['y_true'], naive_pool['y_pred'], f['y_pred'], horizon=24)
        dm_stat, dm_pval, nw_lag = dm.statistic, dm.p_value, dm.newey_west_lag
    metric_rows.append({
        'regime': pooled_label, 'model': model_name, 'n_hours': len(f),
        'MAE (EUR/MWh)': m, 'sMAPE (%)': s, 'rMAE vs naive': r,
        'DM stat vs naive': dm_stat, 'DM p-value vs naive': dm_pval, 'NW lag': nw_lag,
    })

# Pooled coverage (derived)
for model_name in MODELS_LIST:
    f = forecasts[(pooled_label, model_name)]
    by_day = f['y_pred'].groupby(f.index.date).apply(lambda s: int(s.notna().sum()))
    coverage_rows.append({
        'regime': pooled_label, 'model': model_name,
        'days in panel':                  int(len(by_day)),
        'full days predicted':            int((by_day == 24).sum()),
        'partial-hour days in panel':     int(((by_day > 0) & (by_day < 24)).sum()),
        'skipped (lag missing or NaN)':   int((by_day == 0).sum()),
    })

robust = pd.DataFrame(metric_rows)
coverage = pd.DataFrame(coverage_rows)
print(f'rows: metrics={len(robust)}, coverage={len(coverage)}')


rows: metrics=16, coverage=16


## 8. Coverage diagnostic — how many days were actually predicted?

`LEAR.predict` skips test days that are not complete 24-hour days in
`test_df` (e.g. after the strict `.dropna()` over the v8 unsafe
columns leaves some calendar days with 22 or 23 hours). The uniform
filter (`fix(model)` in `3d3a912`) makes this skip identical across
configurations, so the `full days predicted` column below is the same
for every model within a regime — any divergence means the filter
regressed.


In [7]:
coverage_pivot = coverage.pivot_table(
    index='regime', columns='model', values='full days predicted', aggfunc='first'
).reindex(ORDER)[MODELS_LIST]
coverage_pivot


model,naive,LEAR ar-only,LEAR demand+wind,LEAR demand+solar+wind
regime,,,,
"2022 (H2, crisis)",162,147,147,147
2023 (full year),345,333,333,333
2024 (full year),341,335,335,335
2023-2024 pooled,686,668,668,668


In [8]:
coverage.to_csv(COVERAGE_PATH, index=False)
print(f'coverage CSV -> {COVERAGE_PATH}')
coverage


coverage CSV -> C:\Users\Carlo\Desktop\Projects\Mibel_forecasting\reports\diagnostics\lear_robustness_dropped_days_2026_05.csv


,regime,model,days in panel,full days predicted,partial-hour days in panel,skipped (lag missing or NaN)
0,"2022 (H2, crisis)",naive,184,162,22,0
1,"2022 (H2, crisis)",LEAR ar-only,184,147,0,37
2,"2022 (H2, crisis)",LEAR demand+wind,184,147,0,37
3,"2022 (H2, crisis)",LEAR demand+solar+wind,184,147,0,37
4,2023 (full year),naive,365,345,20,0
5,2023 (full year),LEAR ar-only,365,333,0,32
6,2023 (full year),LEAR demand+wind,365,333,0,32
7,2023 (full year),LEAR demand+solar+wind,365,333,0,32
8,2024 (full year),naive,362,341,18,3
9,2024 (full year),LEAR ar-only,362,335,0,27


## 9. Robustness table

MAE, sMAPE and rMAE per (regime, model), with DM p-values against the
seasonal naive in each regime. DM uses Newey-West with the Andrews
(1991) lag rule, floored at `h-1` for the day-ahead horizon `h=24`.


In [9]:
# Re-order rows: ORDER × MODELS_LIST
robust['regime'] = pd.Categorical(robust['regime'], categories=ORDER, ordered=True)
robust['model']  = pd.Categorical(robust['model'],  categories=MODELS_LIST, ordered=True)
robust = robust.sort_values(['regime','model']).reset_index(drop=True)
robust.to_csv(CSV_PATH, index=False, float_format='%.4f')
print(f'metrics CSV -> {CSV_PATH}')
robust.round(4)


metrics CSV -> C:\Users\Carlo\Desktop\Projects\Mibel_forecasting\reports\diagnostics\lear_robustness_2026_05.csv


,regime,model,n_hours,MAE (EUR/MWh),sMAPE (%),rMAE vs naive,DM stat vs naive,DM p-value vs naive,NW lag
0,"2022 (H2, crisis)",naive,4272,32.2478,29.3857,1.0000,NaN,NaN,NaN
1,"2022 (H2, crisis)",LEAR ar-only,4272,20.0136,17.6216,0.6206,7.8490,0.0,23.0
2,"2022 (H2, crisis)",LEAR demand+wind,4272,18.2632,16.0957,0.5663,9.2796,0.0,23.0
3,"2022 (H2, crisis)",LEAR demand+solar+wind,4272,18.2717,15.9824,0.5666,9.2221,0.0,23.0
4,2023 (full year),naive,8663,28.5507,48.5210,1.0000,NaN,NaN,NaN
5,2023 (full year),LEAR ar-only,8663,14.7555,25.8396,0.5168,13.0453,0.0,23.0
6,2023 (full year),LEAR demand+wind,8663,12.8994,24.5560,0.4518,14.4965,0.0,23.0
7,2023 (full year),LEAR demand+solar+wind,8663,13.4149,24.4634,0.4699,13.9714,0.0,23.0
8,2024 (full year),naive,8591,27.4344,75.5185,1.0000,NaN,NaN,NaN
9,2024 (full year),LEAR ar-only,8591,15.3693,52.8736,0.5602,11.4125,0.0,23.0


## 10. Markdown export

Combined markdown report (validity preconditions, coverage, metrics,
honest-reporting flags) at
`reports/diagnostics/lear_robustness_2026_05.md`. The dropped-days
diagnostic is also exported separately as a CSV (§8) so it is easy to
reload without re-running the notebook.


In [10]:
LIMIT_HIGH_RMAE = 0.6
EXOG_MARGINAL_DELTA = 0.01

weak_rows = robust[(robust['model'] != 'naive') &
                   (robust['rMAE vs naive'] > LIMIT_HIGH_RMAE)]

narrow_exog_gap = []
for regime in ORDER:
    sub = robust[robust['regime'] == regime]
    if sub.empty:
        continue
    ar = float(sub[sub['model'] == 'LEAR ar-only']['rMAE vs naive'].iloc[0])
    dw = float(sub[sub['model'] == 'LEAR demand+wind']['rMAE vs naive'].iloc[0])
    if abs(ar - dw) < EXOG_MARGINAL_DELTA:
        narrow_exog_gap.append((regime, ar, dw))

lines = [
    '# LEAR robustness across regimes (2022 H2 / 2023 / 2024)',
    '',
    '> _Auto-generated by `notebooks/03_lear_robustness.ipynb`. Do not hand-edit._',
    '',
    '## Validity preconditions',
    '',
    '- Internal panel is UTC (commit `b38b456`).',
    '- Realised targets are masked before reaching `predict` (commit `00a0aaf`).',
    '- Partial test days (< 24 hours after `.dropna()`) are skipped uniformly across LEAR variants (commits `89cd2db`, `3d3a912`). See dropped-day diagnostic below.',
    '- Exogenous columns used (`es_demand_fc`, `es_wind_fc`, `es_solar_fc`) are documented as day-ahead forecasts in `reports/diagnostics/exogenous_audit_2026_05.md`. Their upstream publication timestamps were not independently re-verified.',
    '- 2022 carries `train_size=180D` (pre-2022 ESIOS cache empty); 2023 and 2024 use the canonical `train_size=365D`. The methodological asymmetry is intentional and disclosed.',
    '',
    '## Coverage diagnostic',
    '',
    'Test days actually predicted vs. partial / skipped, per (regime, model).',
    'With the uniform partial-day filter, the `full days predicted` column is identical across models within a regime — any divergence would indicate the filter has regressed.',
    '',
    '| regime | model | days in panel | full days predicted | partial-hour days in panel | skipped (lag missing or NaN) |',
    '| --- | --- | ---: | ---: | ---: | ---: |',
]

for _, r in coverage.iterrows():
    lines.append(
        f"| {r['regime']} | {r['model']} | {int(r['days in panel'])} "
        f"| {int(r['full days predicted'])} | {int(r['partial-hour days in panel'])} "
        f"| {int(r['skipped (lag missing or NaN)'])} |"
    )

lines += [
    '',
    '## Results',
    '',
    '| regime | model | n hours | MAE (EUR/MWh) | sMAPE (%) | rMAE vs naive | DM stat | DM p-value | NW lag |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |',
]

def _fmt(x: float, nd: int = 4) -> str:
    if pd.isna(x):
        return ''
    return f'{x:.{nd}f}'

for _, r in robust.iterrows():
    lines.append(
        f"| {r['regime']} | {r['model']} | {int(r['n_hours'])} "
        f"| {_fmt(r['MAE (EUR/MWh)'], 3)} | {_fmt(r['sMAPE (%)'], 2)} | {_fmt(r['rMAE vs naive'])} "
        f"| {_fmt(r['DM stat vs naive'], 3)} | {_fmt(r['DM p-value vs naive'])} | {_fmt(r['NW lag'], 0)} |"
    )

lines += ['', '## Honest reporting', '']
if not weak_rows.empty:
    lines += ['**rMAE above 0.60** (model effectively underperforming the naive):', '']
    for _, row in weak_rows.iterrows():
        lines.append(
            f"- {row['regime']} / {row['model']}: rMAE = {row['rMAE vs naive']:.4f}"
        )
    lines.append('')
else:
    lines += [
        'No (regime, model) pair has `rMAE > 0.60` — every LEAR variant beats the seasonal naive comfortably in every window tested. This is consistent with the structural-naive-failure argument of notebook 02 §6 (MIBEL post-2023 has many zero-price mid-day hours that the week-old naive cannot anticipate).',
        '',
    ]

if narrow_exog_gap:
    lines += [
        '**Exogenous contribution is marginal.** In the following regimes, `LEAR ar-only` matches `LEAR demand+wind` to within `ΔrMAE < 0.01`:',
        '',
    ]
    for regime, ar, dw in narrow_exog_gap:
        lines.append(f"- {regime}: ar-only = {ar:.4f}, demand+wind = {dw:.4f}")
    lines += [
        '',
        'This is **a distinct finding from Demir (2019)** on Belgian DAM, where the exogenous block added a measurable improvement. In modern Iberian DAM the AR-lag block alone captures most of the predictable structure once daily recalibration is applied; the demand and wind day-ahead forecasts add only marginal value at this horizon.',
        '',
    ]
else:
    lines += [
        'In every regime tested `LEAR demand+wind` beats `LEAR ar-only` by at least `ΔrMAE >= 0.01`, i.e. the exogenous block contributes measurable predictive content. Consistent with the Demir (2019) Belgian-DAM finding.',
        '',
    ]

lines += [
    '## Provenance',
    '',
    '- Notebook: `notebooks/03_lear_robustness.ipynb`',
    '- Underlying metrics: `reports/diagnostics/lear_robustness_2026_05.csv`',
    '- Dropped-day diagnostic: `reports/diagnostics/lear_robustness_dropped_days_2026_05.csv`',
    f'- Date generated: {pd.Timestamp.now().strftime("%Y-%m-%d")}',
    '',
]

MD_PATH.write_text('\n'.join(lines), encoding='utf-8')
print(f'markdown -> {MD_PATH}')
print()
print('\n'.join(lines[:32]))


markdown -> C:\Users\Carlo\Desktop\Projects\Mibel_forecasting\reports\diagnostics\lear_robustness_2026_05.md

# LEAR robustness across regimes (2022 H2 / 2023 / 2024)

> _Auto-generated by `notebooks/03_lear_robustness.ipynb`. Do not hand-edit._

## Validity preconditions

- Internal panel is UTC (commit `b38b456`).
- Realised targets are masked before reaching `predict` (commit `00a0aaf`).
- Partial test days (< 24 hours after `.dropna()`) are skipped uniformly across LEAR variants (commits `89cd2db`, `3d3a912`). See dropped-day diagnostic below.
- Exogenous columns used (`es_demand_fc`, `es_wind_fc`, `es_solar_fc`) are documented as day-ahead forecasts in `reports/diagnostics/exogenous_audit_2026_05.md`. Their upstream publication timestamps were not independently re-verified.
- 2022 carries `train_size=180D` (pre-2022 ESIOS cache empty); 2023 and 2024 use the canonical `train_size=365D`. The methodological asymmetry is intentional and disclosed.

## Coverage diagnostic

Test days ac

## 11. Error analysis (LEAR demand+wind on 2024 full year)

Zoomed-in diagnostics on the canonical Lago configuration in the most
recent full-year regime. Every table reports MAE in €/MWh and rMAE
relative to that same regime's seasonal-naive baseline.


In [11]:
FOCUS_REGIME, FOCUS_MODEL = '2024 (full year)', 'LEAR demand+wind'
focus = forecasts[(FOCUS_REGIME, FOCUS_MODEL)]
naive_focus = forecasts[(FOCUS_REGIME, 'naive')]

merged = focus.copy()
merged['y_naive'] = naive_focus['y_pred']
merged = merged.dropna()
merged['err']       = (merged['y_true'] - merged['y_pred']).abs()
merged['err_naive'] = (merged['y_true'] - merged['y_naive']).abs()
merged['hour']    = merged.index.hour
merged['month']   = merged.index.month
merged['weekday'] = merged.index.dayofweek < 5
len(merged), merged['err'].mean(), merged['err_naive'].mean()


(8040, np.float64(14.010729359810899), np.float64(27.974398009950246))

### 11.1 By hour of day


In [12]:
def _bucket_table(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        'n':           len(g),
        'MAE (model)': g['err'].mean(),
        'MAE (naive)': g['err_naive'].mean(),
        'rMAE':         g['err'].mean() / g['err_naive'].mean(),
    })

by_hour_tbl = merged.groupby('hour', as_index=True).apply(_bucket_table)
by_hour_tbl.round(3)


,n,MAE (model),MAE (naive),rMAE
hour,,,,
0,335.0,4.118,28.346,0.145
1,335.0,5.840,29.078,0.201
2,335.0,6.813,29.947,0.228
3,335.0,7.382,29.800,0.248
4,335.0,8.299,29.263,0.284
5,335.0,11.975,28.692,0.417
6,335.0,14.817,28.311,0.523
7,335.0,17.563,28.872,0.608
8,335.0,17.707,28.681,0.617


### 11.2 By month


In [13]:
by_month_tbl = merged.groupby('month', as_index=True).apply(_bucket_table)
by_month_tbl.round(3)


,n,MAE (model),MAE (naive),rMAE
month,,,,
1,744.0,10.588,29.501,0.359
2,696.0,13.996,25.034,0.559
3,696.0,19.362,22.192,0.872
4,504.0,18.561,9.440,1.966
5,504.0,10.796,20.734,0.521
6,720.0,14.376,30.662,0.469
7,744.0,10.917,21.117,0.517
8,744.0,11.935,18.509,0.645
9,720.0,13.292,38.568,0.345


### 11.3 Weekday vs weekend


In [14]:
by_dowtype = merged.groupby('weekday', as_index=True).apply(_bucket_table)
by_dowtype.index = by_dowtype.index.map({True: 'weekday', False: 'weekend'})
by_dowtype.round(3)


,n,MAE (model),MAE (naive),rMAE
weekday,,,,
weekend,2208.0,17.012,29.563,0.575
weekday,5832.0,12.874,27.373,0.470


### 11.4 Extreme prices (top 5% / bottom 5%)


In [15]:
q05 = merged['y_true'].quantile(0.05)
q95 = merged['y_true'].quantile(0.95)
buckets = {
    f'bottom 5% (<= {q05:.2f})': merged[merged['y_true'] <= q05],
    f'top 5% (>= {q95:.2f})':    merged[merged['y_true'] >= q95],
    'middle 90%':                 merged[(merged['y_true'] > q05) & (merged['y_true'] < q95)],
}
extreme = pd.DataFrame({k: _bucket_table(v) for k, v in buckets.items()}).T
extreme.round(3)


,n,MAE (model),MAE (naive),rMAE
bottom 5% (<= 0.00),677.0,20.903,12.359,1.691
top 5% (>= 137.61),402.0,20.763,33.844,0.613
middle 90%,6961.0,12.950,29.154,0.444


### 11.5 Near-zero prices (< 1 €/MWh)


In [16]:
near_zero = merged[merged['y_true'] < 1.0]
positive  = merged[merged['y_true'] >= 1.0]
near_zero_tbl = pd.DataFrame({
    f'price_es < 1 (n={len(near_zero)})':  _bucket_table(near_zero),
    f'price_es >= 1 (n={len(positive)})':  _bucket_table(positive),
}).T
near_zero_tbl.round(3)


,n,MAE (model),MAE (naive),rMAE
price_es < 1 (n=905),905.0,20.602,14.501,1.421
price_es >= 1 (n=7135),7135.0,13.175,29.683,0.444


### 11.6 High solar generation (top quartile of `es_solar_fc`)

If the forecasted solar generation column is absent from the panel
the cell logs that and skips the analysis (no silent NaN tables).


In [17]:
if 'es_solar_fc' in df.columns:
    solar = df.loc[merged.index, 'es_solar_fc']
    q75 = solar.quantile(0.75)
    merged_solar = merged.join(solar.rename('solar'))
    high_solar = merged_solar[merged_solar['solar'] >= q75]
    low_solar  = merged_solar[merged_solar['solar'] <  q75]
    solar_tbl = pd.DataFrame({
        f'top quartile (>= {q75:.0f} MW, n={len(high_solar)})': _bucket_table(high_solar),
        f'rest (< {q75:.0f} MW, n={len(low_solar)})':           _bucket_table(low_solar),
    }).T
    solar_tbl_out = solar_tbl.round(3)
else:
    solar_tbl_out = 'es_solar_fc not present in panel — analysis skipped.'
solar_tbl_out


,n,MAE (model),MAE (naive),rMAE
"top quartile (>= 10562 MW, n=2010)",2010.0,16.525,24.526,0.674
"rest (< 10562 MW, n=6030)",6030.0,13.172,29.124,0.452


## 12. Honest reporting and validity conditions

The numbers in §9 and §11 are valid only under the preconditions
listed in §2 and the partial-day-skipping rule confirmed in §8. The
markdown export in §10 also runs two automated checks:

- **rMAE > 0.60** flagged explicitly per (regime, model) — a model
  scoring above 0.60 effectively underperforms the seasonal naive on
  that regime;
- **Marginal exogenous contribution** flagged when `LEAR ar-only`
  matches `LEAR demand+wind` within `ΔrMAE < 0.01` — distinct from
  Demir (2019) on Belgian DAM if it fires.

**On the headline rMAE values.** Across the regimes tested, every
LEAR variant beats the seasonal naive by a large margin (rMAE ≪ 1).
This is *not* a 'better than Lago 2021' result — see notebook 02 §6
for the structural reason (heavy solar penetration → cheap-to-beat
weekly seasonal baseline in MIBEL 2023-2024). The robustness
question this notebook answers is whether the **relative ordering
between LEAR variants is stable across regimes**, and whether any
regime breaks the model outright.

**Next** — notebook 04 will introduce the Demir (2019) technical
indicators on top of the demand+wind baseline and test whether they
shift these conclusions on the same regime windows.
